# Actividad Extra: Analisis de Datos de Vuelos

## Objetivos
- Leer y explorar un dataset de vuelos
- Crear columnas derivadas a partir de datos existentes
- Aplicar pivoting para analisis multidimensional
- Crear funciones personalizadas de agregacion
- Construir un pipeline de Machine Learning con Spark MLlib

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("actividad_vuelos") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

---
## Ejercicio 1: Lectura y exploracion del dataset

Lee el archivo `flights.csv`, muestra el schema y las primeras 5 filas, y cuenta el total de registros.

**Columnas del dataset:** year, month, day, dep_time, dep_delay, arr_time, arr_delay, carrier, tailnum, flight, origin, dest, air_time, distance, hour, minute

In [ ]:
# =============================================================
# EJERCICIO 1: Lectura y exploracion
# =============================================================

# Leer el archivo CSV
df = spark.read.csv("/home/jovyan/datos/flights.csv", header=True, inferSchema=True)

# Mostrar el schema
df.printSchema()

# Mostrar las primeras 5 filas
df.show(5)

# Contar el total de registros
print(f"Total de registros: {df.count()}")
print(f"Columnas: {df.columns}")

> **Nota docente:** Este dataset contiene columnas en minusculas (year, month, day, dep_delay, etc.). Es importante que los alumnos verifiquen los nombres exactos con `printSchema()` antes de usarlos en las consultas. El parametro `inferSchema=True` permite que Spark detecte automaticamente los tipos de datos, pero en produccion es preferible definir el schema explicitamente para mayor control y rendimiento.

---
## Ejercicio 2: Crear columna de fecha

Combina las columnas `year`, `month` y `day` para crear una nueva columna `fecha` de tipo Date.

**Pista:** Usa `F.to_date(F.concat_ws("-", "year", "month", "day"))` para concatenar y convertir a fecha.

In [ ]:
# =============================================================
# EJERCICIO 2: Crear columna de fecha
# =============================================================

# Crear la columna fecha combinando year, month, day
df = df.withColumn("fecha", F.to_date(F.concat_ws("-", "year", "month", "day")))

# Verificar el resultado
df.select("year", "month", "day", "fecha").show(10)

> **Nota docente:** `F.concat_ws("-", ...)` concatena columnas usando un separador (en este caso "-"). Luego `F.to_date()` parsea el string resultante (ej: "2014-1-1") a un tipo Date. Si el formato del string no es reconocido automaticamente, se puede especificar el formato como segundo argumento: `F.to_date(..., "yyyy-M-d")`. Los alumnos pueden confundir `concat_ws` con `concat`: la diferencia es que `concat_ws` inserta un separador entre los valores.

---
## Ejercicio 3: Dia de la semana

A partir de la columna `fecha`, crea una columna `dia_semana` con el numero de dia (1=Domingo...7=Sabado) y luego mapea a nombre legible.

**Pista:** Usa `F.dayofweek("fecha")` y luego `F.when()` encadenado para mapear numeros a nombres.

In [ ]:
# =============================================================
# EJERCICIO 3: Dia de la semana
# =============================================================

# Crear columna con numero de dia de la semana
df = df.withColumn("dia_semana", F.dayofweek("fecha"))

# Mapear numero a nombre del dia
df = df.withColumn("dia_nombre",
    F.when(F.col("dia_semana") == 1, "Domingo")
    .when(F.col("dia_semana") == 2, "Lunes")
    .when(F.col("dia_semana") == 3, "Martes")
    .when(F.col("dia_semana") == 4, "Miercoles")
    .when(F.col("dia_semana") == 5, "Jueves")
    .when(F.col("dia_semana") == 6, "Viernes")
    .when(F.col("dia_semana") == 7, "Sabado")
    .otherwise("Desconocido")
)

# Verificar
df.select("fecha", "dia_semana", "dia_nombre").show(10)

> **Nota docente:** `F.dayofweek()` retorna un entero del 1 (Domingo) al 7 (Sabado) segun la convencion de Spark (basada en Java Calendar). Esto puede confundir a alumnos que esperan que Lunes sea 1 (convencion ISO). El patron `F.when().when()...otherwise()` es el equivalente de un `CASE WHEN` en SQL. Alternativa mas elegante: crear un diccionario y usar `F.create_map()` para el mapeo, o usar `F.date_format("fecha", "EEEE")` para obtener el nombre directamente en ingles.

---
## Ejercicio 4: Pivoting - Vuelos por dia de semana y aerolinea

Crea una tabla pivoteada que muestre la cantidad de vuelos por aerolinea (`carrier`) para cada dia de la semana.

**Pista:** Usa `.groupBy("carrier").pivot("dia_nombre").count()`

In [ ]:
# =============================================================
# EJERCICIO 4: Pivoting
# =============================================================

# Tabla pivoteada: vuelos por aerolinea y dia de semana
df_pivot = df.groupBy("carrier").pivot("dia_nombre").count()

# Mostrar resultado completo
df_pivot.show(truncate=False)

> **Nota docente:** El metodo `pivot()` transforma valores unicos de una columna en columnas individuales, creando una tabla cruzada. Es una operacion costosa porque Spark necesita primero recolectar los valores unicos de la columna pivoteada. Para mejorar el rendimiento, se puede pasar una lista explicita de valores: `.pivot("dia_nombre", ["Lunes", "Martes", ...])`. Los valores `null` en el resultado indican que no hay datos para esa combinacion. Error comun: los alumnos olvidan que `pivot()` va entre `groupBy()` y la funcion de agregacion.

---
## Ejercicio 5: Retrasos medios por periodo del dia

Crea una funcion que clasifique los vuelos segun el periodo del dia basandose en la columna `hour`:
- 06-11: Manana
- 12-17: Tarde
- 18-23: Noche
- 00-05: Madrugada

Luego calcula el promedio de `dep_delay` por cada periodo.

In [ ]:
# =============================================================
# EJERCICIO 5: Retrasos medios por periodo del dia
# =============================================================

# Crear columna de periodo del dia basada en la hora
df = df.withColumn("periodo",
    F.when((F.col("hour") >= 6) & (F.col("hour") <= 11), "Manana")
    .when((F.col("hour") >= 12) & (F.col("hour") <= 17), "Tarde")
    .when((F.col("hour") >= 18) & (F.col("hour") <= 23), "Noche")
    .otherwise("Madrugada")
)

# Calcular retraso promedio por periodo
df_periodos = df.groupBy("periodo").agg(
    F.avg("dep_delay").alias("retraso_promedio"),
    F.count("*").alias("num_vuelos")
).orderBy("retraso_promedio", ascending=False)

df_periodos.show()

> **Nota docente:** Este ejercicio combina la creacion de columnas derivadas con agregaciones. La columna `hour` del dataset ya contiene la hora programada de salida como entero (0-23), lo que facilita la clasificacion. Los alumnos deben recordar que las condiciones multiples en `F.when()` requieren parentesis alrededor de cada comparacion y el operador `&` (AND logico a nivel de columna). Error comun: usar `and` de Python en lugar de `&` de Spark, lo cual genera un error. Los resultados tipicamente muestran que los vuelos nocturnos acumulan mas retraso.

---
## Ejercicio 6: Pipeline de Machine Learning

Construye un pipeline de ML para predecir si un vuelo tendra un retraso significativo (>15 min).

**Pasos:**
1. Usa `StringIndexer` para codificar `carrier`
2. Usa `VectorAssembler` para combinar features: carrier_idx, month, distance, hour
3. Usa `Binarizer` para crear la variable objetivo: dep_delay > 15 -> 1, sino -> 0
4. Entrena un `DecisionTreeClassifier`
5. Evalua la precision (accuracy)

In [ ]:
# =============================================================
# EJERCICIO 6: Pipeline de Machine Learning
# =============================================================

from pyspark.ml.feature import StringIndexer, VectorAssembler, Binarizer
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

# Preparar datos: eliminar nulls en dep_delay
df_ml = df.na.drop(subset=["dep_delay", "carrier", "month", "distance", "hour"])

# Crear variable objetivo: retraso > 15 minutos
df_ml = df_ml.withColumn("label",
    F.when(F.col("dep_delay") > 15, 1.0).otherwise(0.0)
)

# StringIndexer para codificar carrier
indexer = StringIndexer(inputCol="carrier", outputCol="carrier_idx", handleInvalid="skip")

# VectorAssembler para combinar features
assembler = VectorAssembler(
    inputCols=["carrier_idx", "month", "distance", "hour"],
    outputCol="features"
)

# DecisionTreeClassifier
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=5)

# Pipeline
pipeline = Pipeline(stages=[indexer, assembler, dt])

# Dividir en train y test
train_data, test_data = df_ml.randomSplit([0.7, 0.3], seed=42)

print(f"Datos de entrenamiento: {train_data.count()}")
print(f"Datos de prueba: {test_data.count()}")

# Entrenar el modelo
modelo = pipeline.fit(train_data)

# Predecir
predicciones = modelo.transform(test_data)

# Evaluar accuracy
evaluador = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluador.evaluate(predicciones)
print(f"\nAccuracy del modelo: {accuracy:.4f}")

# Mostrar algunas predicciones
predicciones.select("carrier", "month", "distance", "hour", "dep_delay", "label", "prediction").show(10)

> **Nota docente:** Este ejercicio introduce el concepto de Pipeline en Spark MLlib. Puntos clave a resaltar: (1) `StringIndexer` convierte variables categoricas a numericas, necesario porque los algoritmos de ML requieren inputs numericos. (2) `VectorAssembler` combina multiples columnas en un unico vector de features, formato requerido por los clasificadores de Spark. (3) El `Pipeline` encadena transformaciones y el modelo en un flujo reproducible. (4) `handleInvalid="skip"` en StringIndexer evita errores con valores no vistos en entrenamiento. (5) La precision puede no ser la mejor metrica si las clases estan desbalanceadas; se podria usar AUC o F1-score. Los alumnos avanzados pueden experimentar con otros clasificadores como RandomForest o GBT.

---
## Resumen

En esta actividad aprendimos:

1. **Lectura y exploracion** de un dataset de vuelos
2. **Creacion de columnas derivadas** con `withColumn`, `to_date`, `concat_ws`
3. **Mapeo de valores** con `F.when()` encadenado
4. **Pivoting** con `groupBy().pivot().count()`
5. **Funciones de agregacion** personalizadas por periodo
6. **Pipeline ML** con StringIndexer, VectorAssembler y DecisionTreeClassifier

In [ ]:
# Detener SparkSession
spark.stop()
print("SparkSession detenida correctamente.")